# 0 || Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import numpy as np
import pandas as pd
from pandas.api.types import is_integer_dtype
from pandas.tseries.holiday import USFederalHolidayCalendar

import sklearn
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning

import xgboost
from xgboost import XGBRegressor

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset


In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
torch 2.9.1


# 1 || Data Processing and Feature Extraction

## 1.1 : Load the DataFrames

In [4]:
# # Construct raw energy and weather datasets.

# DFDICT = {
#     subdir.name: { 
#         csv.stem: pd.read_csv(csv, dtype="string", low_memory=False)
#         for csv in subdir.glob("*.csv") 
#     }
#     for subdir in (Path.cwd() / "data").iterdir() if subdir.is_dir()
# }

# DFE = ( 
#     DFDICT['electricity']['FPL'][ECOLS]
# )
# DFW = pd.concat(
#     [DFDICT['weather']['MIA_20'+str(y)][WCOLS] for y in range(15, 26)],
#     ignore_index=True
# )


In [5]:
# AUTOREGRESSIVE (LAG) FEATURES

lags = sorted(set(
    list(range(1, 7)) + [12, 18]
    + list(range(24, 27)) + [36, 48]
    + [24*i for i in range(3,7)]
    + [24*7*i for i in range(1,5)]
))
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]


In [6]:
# CALENDAR FEATURES

# Raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]

# Collate all calendar features
calendarFeatures = (
    intDateTimeFeatures
    + hourFourierFeatures
    + dayFourierFeatures
    + hourDummyFeatures
    + dayDummyFeatures
    + monthDummyFeatures
)


In [7]:
# ENERGY FEATURES

energyFeatures = [
    "Adjusted net generation",
    "Adjusted total interchange",
    "FPC", "FMPP", "SOCO", "TEC",
    "JEA", "SEC", "HST", "GVL",
]


In [8]:
# WEATHER FEATURES

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'NA']
directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "VRB"]

weatherFeatures = ([
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed"]
    + [f"HourlySkyConditions_Flag_{code}" for code in skyCodes]
    + [f"HourlyWindDirection_Flag_{d}" for d in directions]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)


In [9]:
# Define other convenient feature variables.

target = "Adjusted demand"
allFeatures = (
    ['t', target]
    + lagFeatures 
    + calendarFeatures 
    + energyFeatures 
    + weatherFeatures
)
someFeatures = (
    ['t', target]
    + lagFeatures[:1] 
    + hourFourierFeatures[:2] + dayFourierFeatures[:2] + dayDummyFeatures[:1]
    + energyFeatures[:2] 
    + weatherFeatures[:1]
)


# 2. Model Building and Training

In [10]:
DFtrain = pd.read_pickle("data/clean/train/DFtrain_block004.pkl")[someFeatures]
DFtest = pd.read_pickle("data/clean/test/DFtest_block011.pkl")[someFeatures]
for c in DFtest.columns: print(c, DFtest[c].dtype)

t datetime64[ns]
Adjusted demand Int32
Adjusted demand -1 hr Int32
sin(Hour) float32
cos(Hour) float32
sin(DayOfYear) float32
cos(DayOfYear) float32
Day_Flag_Weekend bool
Adjusted net generation Int32
Adjusted total interchange Int16
HourlyDryBulbTemperature Int16


In [11]:
X_train = DFtrain.iloc[:, 2:].to_numpy(dtype=np.float32)
y_train = DFtrain[target].to_numpy(dtype=np.float32)
X_test  = DFtest.iloc[:, 2:].to_numpy(dtype=np.float32)
y_test  = DFtest[target].to_numpy(dtype=np.float32)

print("\nX_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)
print("\nTrain range: ", DFtrain["t"].min(), " - ", DFtrain["t"].max())
print("Test range:  ", DFtest["t"].min(),  " - ", DFtest["t"].max())

X_train: (1534, 9) y_train: (1534,)
X_test:  (895, 9)  y_test:  (895,)
Train range:  2015-08-29 02:00:00  -  2015-10-31 23:00:00
Test range:   2024-08-31 14:00:00  -  2024-10-07 20:00:00


In [ ]:
# LSTM time-series regressor:
# 1. Scale data
# 2. Build sliding-window sequences
# 3. Wrap in DataLoaders
# 4. Define LSTM model
# 5. Train and evaluate (RMSE in original units)

# --- 1. Settings and scaling ---

seq_len = 24  # use past 24 hours to predict the next hour

# Scale input features using training set statistics
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled  = x_scaler.transform(X_test)

# Scale target variable using training set statistics
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()


# --- 2. Build sequences for LSTM ---

def make_sequences(X, y, seq_len):
    """
    Turn a time-ordered dataset into (sequence, target) pairs for an LSTM.

    Given features X and targets y, this function creates overlapping windows
    of length `seq_len`, where each window predicts the next value of y.
    """
    # Ensure float32 arrays and 1D target
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    X_seq_list = []
    y_seq_list = []

    # Slide a window of length `seq_len` over the data
    for start in range(len(X) - seq_len):
        end = start + seq_len
        X_seq_list.append(X[start:end])   # window of features
        y_seq_list.append(y[end])         # value immediately after window

    # Stack into tensors of shape:
    #   X_seq: (num_samples, seq_len, num_features)
    #   y_seq: (num_samples,)
    X_seq = np.stack(X_seq_list)
    y_seq = np.array(y_seq_list)

    return torch.tensor(X_seq), torch.tensor(y_seq)


# Apply sequence builder to training and test splits
X_train_seq, y_train_seq = make_sequences(X_train_scaled, y_train_scaled, seq_len)
X_test_seq,  y_test_seq  = make_sequences(X_test_scaled,  y_test_scaled,  seq_len)


# --- 3. DataLoaders ---

# Wrap sequence tensors into Dataset objects
train_dataset = TensorDataset(X_train_seq, y_train_seq)
test_dataset  = TensorDataset(X_test_seq,  y_test_seq)

# Create DataLoaders for mini-batch training and evaluation
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False)


# --- 4. Device selection ---

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# --- 5. LSTM model definition ---

class LSTMRegressor(nn.Module):
    """
    Simple LSTM-based regressor that maps a sequence of feature vectors
    to a single scalar prediction (next-hour demand).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()

        # LSTM processes the sequence and produces hidden states over time
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0,
        )

        # Final linear layer maps the last hidden state to a scalar
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch_size, seq_len, num_features)
        lstm_out, _ = self.lstm(x)            # (batch_size, seq_len, hidden_size)
        last_hidden = lstm_out[:, -1, :]      # take hidden state at last time step
        output = self.fc(last_hidden)         # (batch_size, 1)
        return output.squeeze(-1)             # (batch_size,)


# Instantiate model, optimizer, and loss (MSE on scaled targets)
model = LSTMRegressor(input_size=X_train_seq.shape[2]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print(model, '\n\n')


# --- 6. Training loop and periodic evaluation ---

n_epochs = 10
rmse = None           # will store last test RMSE
y_true = y_pred = None  # will store last test targets and predictions

for epoch in range(1, n_epochs + 1):
    
    # Put model in training mode
    model.train()
    batch_losses = []

    # Iterate over mini-batches
    for X_batch, y_batch in train_loader:
        
        # Move batch to device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass and loss computation
        optimizer.zero_grad()
        y_pred_batch = model(X_batch)
        loss = criterion(y_pred_batch, y_batch)

        # Backward pass and parameter update
        loss.backward()
        optimizer.step()

        # Track batch loss for this epoch
        batch_losses.append(loss.item())

    # Average training loss over all batches
    train_mse = float(np.mean(batch_losses))

    # Evaluate on test set every 5 epochs (and at epoch 1)
    if epoch % 5 == 0:
        
        # Put model in evaluation mode (disables dropout, etc.)
        model.eval()
        y_pred_scaled_list = []
        y_true_scaled_list = []

        # Collect predicted and true values in scaled space
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_pred_scaled_list.append(model(X_batch).cpu().numpy())
                y_true_scaled_list.append(y_batch.numpy())

        # Concatenate all mini-batch outputs
        y_pred_scaled = np.concatenate(y_pred_scaled_list)
        y_true_scaled = np.concatenate(y_true_scaled_list)

        # Convert predictions and targets back to original demand units
        y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        y_true = y_scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).ravel()

        # Compute RMSE on the original scale
        rmse = root_mean_squared_error(y_true, y_pred)

        # Print both training MSE (scaled) and test RMSE (original units)
        print(
            f"Epoch {epoch:02d} / {n_epochs}  "
            f"Train MSE (scaled): {train_mse:.3f}  "
            f"Test RMSE: {rmse:.1f}"
        )
        
    else:
        # Print only training loss for non-eval epochs
        print(
            f"Epoch {epoch:02d} / {n_epochs}  "
            f"Train MSE (scaled): {train_mse:.3f}"
        )

# --- 7. Final summary ---

# After training, y_pred / y_true / rmse refer to the last evaluation
print(f"\nLSTM RMSE: {rmse:.1f}")
